=======================================================================
aisles table
=======================================================================

starting with Aisle_t where we are checking for white spaces/ lower case & 
will be adding NOT null constraint, will be checking for any duplicate as well

later for future will do primary key restraint on aisle_id and if found we will be putting in a new table

To check for duplicates

In [0]:


%sql
select aisle_id,aisle from
(select  aisle_id,
initcap(REGEXP_REPLACE(TRIM(lower(aisle)), '\\s+', ' ') )AS aisle,
row_number() over (partition by aisle_id order by aisle_id) as rn 
from instacart.bronze_schema.aisles_t 
where aisle_id is not NULL 
 ) where rn!=1;

In [0]:
create table if not exists instacart.silver_schema.aisles_t
(aisle_id int NOT NULL,
aisle_name string NOT NULL);

if duplicate is there then we use window function for selecting the top one.

In [0]:
insert into instacart.silver_schema.aisles_t
select aisle_id,aisle from
(select  aisle_id,
initcap(REGEXP_REPLACE(TRIM(lower(aisle)), '\\s+', ' ') )AS aisle,
row_number() over (partition by aisle_id order by aisle_id) as rn 
from instacart.bronze_schema.aisles_t 
where aisle_id is not NULL 
and aisle is not null
and aisle_id>0
 ) where rn=1
;

In [0]:
select * from instacart.silver_schema.aisles_t

For future uploads, making the constrainsts if required can be used.

In [0]:
alter table instacart.silver_schema.aisles_t 
add constraint aisles_pk PRIMARY KEY (aisle_id);

alter table instacart.silver_schema.aisles_t
add constraint aisles_ck CHECK (aisle_id>0);

alter table instacart.silver_schema.aisles_t
add constraint aisles_uk UNIQUE (aisle_name);

select * from instacart.silver_schema.aisles_t

In [0]:
select aisle_id,count(*) 
from instacart.bronze_schema.aisles_t
group by aisle_id
having count(*)>1;

merge statement we are using for a situation where new data comes for update in the csv file and aisle name is getting updated and if it is not then we need to insert.

In [0]:
merge into instacart.silver_schema.aisles_t t
using (select  aisle_id,
initcap(REGEXP_REPLACE(TRIM(lower(aisle)), '\\s+', ' ') )AS aisle_name 
from instacart.bronze_schema.aisles_t where aisle_id is not NULL
and aisle is not NULL
and aisle_id>0) s
on t.aisle_id=s.aisle_id
when matched then update set t.aisle_name=s.aisle_name
when not matched then insert (aisle_id,aisle_name) values (s.aisle_id,s.aisle_name)

=============================================================
department table
=============================================================


In [0]:
select department_id,department from (
select department_id,
initcap(REGEXP_REPLACE(TRIM(lower(department)), '\\s+', ' ') )AS department,
row_number() over (partition by department_id order by department_id) as rn
 from instacart.bronze_schema.departments_t where department_id is not NULL
and department is not NULL
and department_id>0) where rn!=1;

In [0]:
create table instacart.silver_schema.departments_t (
    department_id int primary key not null,
    department string unique not null
)
comment "silver table for department "

In [0]:
insert into instacart.silver_schema.departments_t
select department_id,department from (
select department_id,
initcap(REGEXP_REPLACE(TRIM(lower(department)), '\\s+', ' ') )AS department,
row_number() over (partition by department_id order by department_id) as rn
 from instacart.bronze_schema.departments_t where department_id is not NULL
and department is not NULL
and department_id>0) where rn=1

In [0]:
select * from instacart.silver_schema.departments_t;

==================================================
product_t
==================================================

In [0]:
select * from instacart.bronze_schema.product_t

In [0]:
select product_id,product_name,aisle_id,department_id from
(select product_id,initcap(REGEXP_REPLACE(TRIM(lower(product_name)), '\\s+', ' ') ) as product_name,cast(aisle_id as int) as aisle_id,cast(department_id as int) as department_id, row_number() over (partition by product_id,p.product_name order by product_id) as rn 
from instacart.bronze_schema.product_t p
 where p.product_id is NOT NULL
and p.product_name is NOT NULL
and p.aisle_id is NOT NULL
and p.department_id is NOT NULL
and p.product_id>0
) where rn!=1 


In [0]:
create table instacart.silver_schema.product_t (
    product_id int primary key not null,
    product_name string unique not null,
    aisle_id int not null,
    department_id int not null,
    FOREIGN KEY (aisle_id) REFERENCES instacart.silver_schema.aisles_t(aisle_id),
    FOREIGN KEY (department_id) REFERENCES instacart.silver_schema.departments_t(department_id)
);

In [0]:
insert into instacart.silver_schema.product_t
select product_id,product_name,aisle_id,department_id from
(select product_id,initcap(REGEXP_REPLACE(TRIM(lower(product_name)), '\\s+', ' ') ) as product_name,cast(aisle_id as int) as aisle_id,cast(department_id as int) as department_id, row_number() over (partition by product_id,p.product_name order by product_id) as rn 
from instacart.bronze_schema.product_t p
 where p.product_id is NOT NULL
and p.product_name is NOT NULL
and p.aisle_id is NOT NULL
and p.department_id is NOT NULL
and p.product_id>0
) where rn=1 ;

In [0]:
select * from instacart.silver_schema.product_t;

========================================
order_t
========================================

In [0]:
select * from instacart.bronze_schema.orders_t 

In [0]:
select count(*) from instacart.bronze_schema.orders_t 
where order_id is not null and user_id is not null and
eval_set is not null and order_number is not null limit 2;



In [0]:
create table if not exists instacart.silver_schema.orders_t
(
order_id int primary key not null,
user_id int not null,
eval_set string not null constraint eval_set_check check (eval_set in ('prior','train','test')),
order_number int not null,
order_dow int constraint dow_check check (dow_check between 0 and 6),
order_hour_of_day int constraint hod_check check (hod_check between 0 and 23 ),
days_since_prior_order int
);

In [0]:
select * from instacart.silver_schema.orders_t where order_id=521107;

In [0]:
insert into instacart.silver_schema.orders_t 
select distinct order_id,user_id,eval_set,order_number,order_dow,order_hour_of_day,
case 
    when days_since_prior_order is null and order_number = 1 then 0
    else days_since_prior_order
end as days_since_prior_order
 from instacart.bronze_schema.orders_t 
where order_id is not null and user_id is not null and
eval_set is not null and order_number is not null 

=============================================================
order_products__prior & order_products__train

In [0]:
select order_id,product_id,add_to_cart_order,reordered,'prior' as eval_src  from instacart.bronze_schema.order_product_prior_t  union all
select order_id,product_id,add_to_cart_order,reordered,'train' as eval_src from instacart.bronze_schema.order_product_train_t  

In [0]:
SELECT order_id, product_id, COUNT(*) 
FROM instacart.silver_schema.order_product_t
GROUP BY order_id, product_id
HAVING COUNT(*) > 1;

In [0]:
create table if not exists instacart.silver_schema.order_product_t
(
order_id int not null references instacart.silver_schema.orders_t(order_id),
product_id int not null references instacart.silver_schema.product_t(product_id),
add_to_cart_order int not null constraint add_to_cart_order_check check (add_to_cart_order > 0),
reordered boolean not null,
eval_src string not null constraint eval_src_check check (eval_src in ('prior','train','test')),
primary key (order_id, product_id)
);

In [0]:
insert into instacart.silver_schema.order_product_t
select order_id,product_id,add_to_cart_order,reordered,'prior' as eval_src  from instacart.bronze_schema.order_product_prior_t  union all
select order_id,product_id,add_to_cart_order,reordered,'train' as eval_src from instacart.bronze_schema.order_product_train_t  